**Version 2:** I updated the qs and bins_list values<br>
**Version 3:** Instead of feature engineering the bins, we can simply increase the max_bin parameter. Credit to [@siukeitin](https://www.kaggle.com/siukeitin) and [@alan1305](https://www.kaggle.com/alan1305) for pointing this out in [this Kaggle discussion](https://www.kaggle.com/competitions/playground-series-s6e3/discussion/679983).

# Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Data Loading

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
print('Train Shape:', train.shape)
display(train.head(3))
print('Test Shape:', test.shape)
display(test.head(3))

In [ ]:
N_SPLITS = 5
SEED = 42

In [ ]:
CATS = ['gender','Partner', 'Dependents', 'SeniorCitizen',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',] 

NUMS = ['tenure', 'MonthlyCharges', 'TotalCharges']

TARGET = 'Churn'

In [ ]:
train[TARGET] = train[TARGET].map({'No': 0, 'Yes': 1})

# Feature Engineering

In [ ]:
combined_df = pd.concat([train, test], axis=0, ignore_index=True)

## DIGIT

In [ ]:
DIGIT = []

target_cols_for_digits = ['tenure', 'MonthlyCharges', 'TotalCharges']

for col in target_cols_for_digits:
    max_val = combined_df[col].abs().max()
    
    if max_val == 0:
        max_int_digits = 1
    else:
        max_int_digits = int(np.log10(max_val)) + 1
    
    for i in range(max_int_digits):
        new_col = f"{col}_digit_pos_{i+1}"
        
        combined_df[new_col] = (combined_df[col] // (10**i)) % 10
        combined_df[new_col] = combined_df[new_col].astype(int)
        
        DIGIT.append(new_col)
    
    if combined_df[col].dtype == 'float64':
        max_dec_digits = 2 
        
        if (combined_df[col] % 1 != 0).any():
            for i in range(1, max_dec_digits + 1):
                new_col = f"{col}_digit_dec_{i}"
                
                combined_df[new_col] = (combined_df[col] * (10**i)).round().astype(int) % 10
                
                DIGIT.append(new_col)

print(f"{len(DIGIT)} Digit Features Created!!")
print(f"Sample features: {DIGIT[:5]} ...")


In [ ]:
for c in CATS:
    combined_df[c] = combined_df[c].astype(str).astype('category')

In [ ]:
train = combined_df.iloc[: len(train)].reset_index(drop=True)
test  = combined_df.iloc[len(train) :].reset_index(drop=True)

# Model Training

In [ ]:
FEATURES = CATS + NUMS + DIGIT

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

In [ ]:
xgb_params = {
    'n_estimators': 20000,      
    'learning_rate': 0.02,
    'max_depth': 3,
    'subsample': 0.8,
    'colsample_bytree':0.8,
    'max_bin':16000,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'n_jobs': -1,
    'random_state': 42,
    'early_stopping_rounds': 200,
    'device': 'cuda',
    
    'enable_categorical': True, 
}

oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))
models = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train[FEATURES], train[TARGET])):
    print(f'Training Fold: {fold+1}')
    
    X_train, y_train = train.loc[train_idx, FEATURES], train.loc[train_idx, TARGET]
    X_val, y_val = train.loc[val_idx, FEATURES], train.loc[val_idx, TARGET]

    
    model = XGBClassifier(**xgb_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=500 
    )
    
    val_preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_preds
    
    test_preds += model.predict_proba(test[FEATURES])[:, 1] / skf.get_n_splits()
    
    fold_auc = roc_auc_score(y_val, val_preds)
    print(f'Fold {fold+1} AUC: {fold_auc:.5f}')
    
    models.append(model)
    print('-' * 30)

overall_auc = roc_auc_score(train[TARGET], oof_preds)
print(f'Overall AUC: {overall_auc:.5f}')


In [ ]:
pd.DataFrame({'id': train.id, TARGET: oof_preds}).to_csv('oof_xgb_bin_dig.csv', index=False)
pd.DataFrame({'id': test.id, TARGET: test_preds}).to_csv('submission.csv', index=False)